# 15.4 — Trio tear sheet, paper and thesis formats

Presentation layer only. This notebook reads one completed `canonical_reports.v1`
bundle and reshapes its hash-inventoried canonical trio table into paper and thesis
derivatives. It does not rebuild market, Factor, or SJM inputs; it does not calculate
financial metrics or select an SJM configuration.

Optional overrides:
- `FINANCE_NOTEBOOK_REPORT_ROOT` — completed `canonical_reports.v1` directory.
- `FINANCE_NOTEBOOK_SOURCE_ROOT` — backward-compatible alias for the same report root.
- `FINANCE_NOTEBOOK_OUTPUT_DIR` — destination for formatted reports; defaults to the clean, validated `reports/validated_notebook_15_4/` subdirectory.
- `FINANCE_NOTEBOOK_REPO_ROOT` — project root when executing a temporary copy outside the repository.

In [1]:
import hashlib
import json
import os
import re
import sys
from pathlib import Path


def _repository_root() -> Path:
    configured = os.environ.get("FINANCE_NOTEBOOK_REPO_ROOT")
    if configured:
        root = Path(configured).expanduser().resolve()
        if not (root / "pyproject.toml").is_file() or not (root / "notebooks").is_dir():
            raise ValueError(
                "FINANCE_NOTEBOOK_REPO_ROOT must name the project directory containing "
                f"pyproject.toml and notebooks/: {root}"
            )
        return root
    for candidate in (Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent):
        if (candidate / "pyproject.toml").is_file() and (candidate / "notebooks").is_dir():
            return candidate.resolve()
    return Path.cwd().resolve()


REPO = _repository_root()
sys.path.insert(0, str(REPO))

import numpy as np
import pandas as pd
from IPython.display import display

from macro_framework.reporting import READER_SCHEMA, validate_report_row
from scripts import build_tear_sheet as bts


TRIO_STEM = "tear_sheet_trio_ext2026"
PROVISIONAL_REPORT_ROOT = REPO / "data" / "provisional_remediation" / "canonical_reports"
DEFAULT_VALIDATED_OUTPUT_DIR = REPO / "reports" / "validated_notebook_15_4"
_DATE_COLUMNS = {
    "start",
    "end",
    "actual_end",
    "anchor",
    "first_return_date",
    "requested_start",
    "requested_end",
    "raw_market_model_start",
    "raw_market_model_end",
}


def _resolve_path(value: str) -> Path:
    path = Path(value).expanduser()
    return path.resolve() if path.is_absolute() else (REPO / path).resolve()


def sha256_file(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


def _report_root() -> Path:
    configured = os.environ.get("FINANCE_NOTEBOOK_REPORT_ROOT") or os.environ.get(
        "FINANCE_NOTEBOOK_SOURCE_ROOT"
    )
    root = _resolve_path(configured) if configured else PROVISIONAL_REPORT_ROOT
    if not root.is_dir():
        raise FileNotFoundError(
            "completed canonical report root is missing: "
            f"{root}. Set FINANCE_NOTEBOOK_REPORT_ROOT to a completed canonical_reports.v1 directory."
        )
    return root


def _output_dir() -> Path:
    configured = os.environ.get("FINANCE_NOTEBOOK_OUTPUT_DIR")
    output = _resolve_path(configured) if configured else DEFAULT_VALIDATED_OUTPUT_DIR
    output.mkdir(parents=True, exist_ok=True)
    if not output.is_dir():
        raise ValueError(f"formatted-report output path is not a directory: {output}")
    return output


def _identity_entry(manifest: dict[str, object], family: str, identity_key: str) -> tuple[str, str]:
    inputs = manifest.get("input_manifests")
    if not isinstance(inputs, dict) or not isinstance(inputs.get(family), dict):
        raise ValueError(f"canonical report manifest has no {family!r} lineage entry")
    entry = inputs[family]
    identity, digest = entry.get(identity_key), entry.get("manifest_sha256")
    if not isinstance(identity, str) or not identity:
        raise ValueError(f"canonical report {family!r} identity is invalid")
    if not isinstance(digest, str) or re.fullmatch(r"[0-9a-f]{64}", digest) is None:
        raise ValueError(f"canonical report {family!r} manifest SHA-256 is invalid")
    return identity, digest


def normalize_table(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    for column in out.columns:
        if column in _DATE_COLUMNS or column.endswith("_date"):
            out[column] = pd.to_datetime(out[column], format="mixed", errors="raise")
    return out


def _scalar_or_none(value):
    if isinstance(value, np.generic):
        value = value.item()
    if value is None or value is pd.NA or value is pd.NaT:
        return None
    return value


def load_completed_trio(root: Path) -> tuple[dict[str, object], pd.DataFrame, Path]:
    manifest_path, marker_path = root / "manifest.json", root / "COMPLETED"
    if not manifest_path.is_file() or not marker_path.is_file():
        raise ValueError(f"{root}: completed canonical report manifest or marker is missing")
    manifest = json.loads(manifest_path.read_text())
    if not isinstance(manifest, dict):
        raise ValueError(f"{manifest_path}: manifest must be a JSON object")
    if manifest.get("schema") != "canonical_reports.v1" or manifest.get("completed") is not True:
        raise ValueError(f"{manifest_path}: expected completed canonical_reports.v1 manifest")
    if f"manifest_sha256={sha256_file(manifest_path)}" not in marker_path.read_text().splitlines():
        raise ValueError(f"{root}: COMPLETED marker does not match manifest bytes")

    tables = manifest.get("tables")
    entry = tables.get(TRIO_STEM) if isinstance(tables, dict) else None
    if not isinstance(entry, dict):
        raise ValueError(f"{manifest_path}: missing inventory entry for {TRIO_STEM}")
    relative_file, expected_sha, expected_rows = entry.get("file"), entry.get("sha256"), entry.get("rows")
    if not isinstance(relative_file, str) or not isinstance(expected_sha, str):
        raise ValueError(f"{manifest_path}: {TRIO_STEM} inventory is incomplete")
    if isinstance(expected_rows, bool) or not isinstance(expected_rows, int) or expected_rows != 3:
        raise ValueError(f"{manifest_path}: {TRIO_STEM} must inventory exactly three rows")
    artifact = (root / relative_file).resolve()
    try:
        artifact.relative_to(root.resolve())
    except ValueError as exc:
        raise ValueError(f"{manifest_path}: {TRIO_STEM} path escapes report root") from exc
    if artifact.suffix != ".parquet" or not artifact.is_file():
        raise ValueError(f"{manifest_path}: {TRIO_STEM} must be an existing Parquet asset")
    if sha256_file(artifact) != expected_sha:
        raise ValueError(f"{artifact}: bytes do not match the canonical report manifest")

    frame = normalize_table(pd.read_parquet(artifact))
    if len(frame) != expected_rows:
        raise ValueError(f"{artifact}: expected {expected_rows} canonical trio rows, found {len(frame)}")
    return manifest, frame, artifact


def validated_trio_rows(frame: pd.DataFrame) -> list[dict[str, object]]:
    if "schema" not in frame or set(frame["schema"]) != {READER_SCHEMA}:
        raise ValueError(f"canonical trio must contain only {READER_SCHEMA} rows")
    rows = [
        validate_report_row({key: _scalar_or_none(value) for key, value in raw.items()})
        for raw in frame.to_dict(orient="records")
    ]
    if len(rows) != 3 or any(row["row_kind"] != "full" for row in rows):
        raise ValueError("canonical trio must contain exactly three full reader rows")
    signatures = {(row["start"], row["end"], row["n_obs"], row["periods_per_year"]) for row in rows}
    if len(signatures) != 1:
        raise ValueError("canonical trio reader rows do not share one performance signature")
    if len({row["cash_benchmark_id"] for row in rows}) != 1 or len({row["currency_basis"] for row in rows}) != 1:
        raise ValueError("canonical trio reader rows do not share cash and currency identities")
    if any("#" not in str(row["source"]) for row in rows):
        raise ValueError("canonical trio reader rows require hash-bound source provenance")
    return rows


REPORT_ROOT = _report_root()
OUT = _output_dir()

## 1. Validate the canonical trio source table

In [2]:
canonical_manifest, trio_loaded, trio_path = load_completed_trio(REPORT_ROOT)
factor_run_id, factor_manifest_sha = _identity_entry(canonical_manifest, "factor_run", "run_id")
overlay_id, sjm_manifest_sha = _identity_entry(canonical_manifest, "sjm_run", "run_id")
snapshot_id, snapshot_manifest_sha = _identity_entry(canonical_manifest, "market_snapshot", "snapshot_id")

reader_rows = validated_trio_rows(trio_loaded)
rows_by_id = {str(row["portfolio_id"]): row for row in reader_rows}
factor_id = "factor_pit_ext2026"
portfolio_ids = set(rows_by_id)
if len(portfolio_ids) != 3 or {factor_id, overlay_id} - portfolio_ids:
    raise ValueError("canonical trio identities must include the Factor PIT and pinned SJM overlay lines")
if not str(rows_by_id[factor_id]["source"]).startswith("scripts/extend_stream_2026.py:"):
    raise ValueError("canonical Factor PIT row is not sourced by the Factor producer")
if not str(rows_by_id[overlay_id]["source"]).startswith(f"sjm_run:{overlay_id}/"):
    raise ValueError("canonical SJM row does not match the pinned SJM run identity")
static_ids = portfolio_ids - {factor_id, overlay_id}
if len(static_ids) != 1:
    raise ValueError("canonical trio must contain exactly one static comparison row")
static_id = static_ids.pop()

PORTFOLIO_ORDER = [static_id, factor_id, overlay_id]
DISPLAY_NAMES = {
    static_id: "Buy-and-hold",
    factor_id: "AI macro-factor",
    overlay_id: "SJM de-risk",
}
trio_reader = pd.DataFrame(reader_rows).set_index("portfolio_id").loc[PORTFOLIO_ORDER].copy()
trio_reader.insert(0, "display_name", [DISPLAY_NAMES[idx] for idx in trio_reader.index])
TRIO_SOURCE_SHA256 = sha256_file(trio_path)
TRIO_SOURCE_PATH = str(trio_path)

for portfolio_id in PORTFOLIO_ORDER:
    for field in ("total_return", "cagr", "ann_vol", "sharpe", "maxdd", "calmar", "ssr_ssr"):
        if not np.isfinite(float(trio_reader.loc[portfolio_id, field])):
            raise ValueError(f"{portfolio_id}: canonical reader metric {field} is not finite")

display(
    trio_reader[
        [
            "display_name", "window_label", "start", "end", "n_obs",
            "total_return", "cagr", "ann_vol", "sharpe", "maxdd", "calmar",
            "ssr_ssr", "cash_benchmark_id", "currency_basis", "source",
        ]
    ]
)
print("canonical report root:", REPORT_ROOT)
print("canonical trio table:", TRIO_SOURCE_PATH)
print("canonical trio table sha256:", TRIO_SOURCE_SHA256)
print("market snapshot:", snapshot_id, snapshot_manifest_sha)
print("factor bundle:", factor_run_id, factor_manifest_sha)
print("sjm run:", overlay_id, sjm_manifest_sha)
print("formatted-report output directory:", OUT)

,display_name,window_label,start,end,n_obs,total_return,cagr,ann_vol,sharpe,maxdd,calmar,ssr_ssr,cash_benchmark_id,currency_basis,source
portfolio_id,,,,,,,,,,,,,,,
static_bh_25pct_2019-01-02,Buy-and-hold,Factor performance window (buy 2019-01-02),2019-01-03,2026-06-30,1845,2.401641,0.177550,0.136449,1.099819,-0.178468,0.994853,0.143575,BIL@provisional_market_total_return_fx_2026-06...,legacy_mixed_local_quotes,scripts/build_tear_sheet.py:static_bh_25pct|ma...
factor_pit_ext2026,AI macro-factor,2019-01-03..2026-06-30,2019-01-03,2026-06-30,1845,1.468075,0.128182,0.092823,1.090813,-0.114053,1.123881,0.133262,BIL@provisional_market_total_return_fx_2026-06...,legacy_mixed_local_quotes,scripts/extend_stream_2026.py:factor_equity_ex...
sjm_crowding_v3_total_return_bil_provisional_devstartfix_20260730T154855Z,SJM de-risk,full 2019-01-03..2026-06-30,2019-01-03,2026-06-30,1845,1.213649,0.111914,0.078984,1.078992,-0.088272,1.267837,0.127859,BIL@provisional_market_total_return_fx_2026-06...,legacy_mixed_local_quotes,sjm_run:sjm_crowding_v3_total_return_bil_provi...


canonical report root: /home/mc/projects/Global_Macro_AI_Factors/data/provisional_remediation/canonical_reports_devstartfix_full_20260730T154855Z
canonical trio table: /home/mc/projects/Global_Macro_AI_Factors/data/provisional_remediation/canonical_reports_devstartfix_full_20260730T154855Z/tables/tear_sheet_trio_ext2026.parquet
canonical trio table sha256: 1380f9aa2ed77e7b86154d95407e0b9fbd32d92a52f6b02af450310e780497cf
market snapshot: provisional_market_total_return_fx_2026-06-30_v1 d56e1d0a17dd3ad05747dd58b829ba3fbf6b56ca46d2f744fc4672ebb8b22a09
factor bundle: factor_ext2026_2019-01-01_2026-06-30_v1 ef38f37f6798773f366b8631a1205cd945e03044d597fd281a9acf94adf33278
sjm run: sjm_crowding_v3_total_return_bil_provisional_devstartfix_20260730T154855Z 070c5f371cc3ea5083c59300f03721fc12f7a5025494668179b9b429748ea0d4
formatted-report output directory: /home/mc/projects/Global_Macro_AI_Factors/reports/validated_notebook_15_4


## 2. Paper and thesis derivatives with source-table provenance

In [3]:

for required in (
    "total_return",
    "cagr",
    "ann_vol",
    "sharpe",
    "sortino",
    "maxdd",
    "calmar",
    "ssr_ssr",
    "raw_market_model_beta",
    "raw_market_model_r2",
    "raw_market_model_intercept_ann_arithmetic",
    "raw_market_model_intercept_t_hac",
    "cash_benchmark_id",
    "currency_basis",
    "source",
):
    assert required in trio_reader.columns, f"canonical trio field missing: {required}"

assert trio_reader["start"].nunique() == 1 and trio_reader["end"].nunique() == 1, "paper/thesis exports require one shared trio window"
window_start = pd.Timestamp(trio_reader["start"].iloc[0]).date()
window_end = pd.Timestamp(trio_reader["end"].iloc[0]).date()
window_days = int(trio_reader["n_obs"].iloc[0])

RETURN_SECTION = [
    ("CAGR", "cagr", "pct"),
    ("Ann. volatility", "ann_vol", "pct"),
    ("Sharpe", "sharpe", "num"),
    ("Max drawdown", "maxdd", "pct"),
    ("Calmar", "calmar", "num"),
    ("SSR", "ssr_ssr", "num"),
]
THESIS_RETURN = [
    ("Total return", "total_return", "pct"),
    *RETURN_SECTION[:3],
    ("Sortino", "sortino", "num"),
    *RETURN_SECTION[3:],
]
RAW_MODEL_SECTION = [
    ("Beta", "raw_market_model_beta", "num"),
    ("R² (raw market model)", "raw_market_model_r2", "num"),
    ("Intercept (ann.)", "raw_market_model_intercept_ann_arithmetic", "pct"),
    ("HAC t(intercept)", "raw_market_model_intercept_t_hac", "num"),
]
WINDOW_SECTION = [
    ("Start", "start", "date"),
    ("End", "end", "date"),
    ("Number of days", "n_obs", "int"),
    ("Cash benchmark", "cash_benchmark_id", "text"),
    ("Currency basis", "currency_basis", "text"),
]
LATEX_NL = "\\\\"
_TEX_TEXT_ESCAPES = {
    "\\": r"\textbackslash{}",
    "&": r"\&",
    "%": r"\%",
    "$": r"\$",
    "#": r"\#",
    "_": r"\_",
    "{": r"\{",
    "}": r"\}",
    "~": r"\textasciitilde{}",
    "^": r"\textasciicircum{}",
}


def tex_escape(value: object) -> str:
    """Render dynamic text safely for a text-mode LaTeX table field."""
    return "".join(_TEX_TEXT_ESCAPES.get(character, character) for character in str(value))


assert LATEX_NL == "\\\\" and len(LATEX_NL) == 2
assert tex_escape(r"\&%$#_{}~^") == r"\textbackslash{}\&\%\$\#\_\{\}\textasciitilde{}\textasciicircum{}"


def format_value(value, kind: str, *, decimal: str = ".") -> str:
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return ""
    if kind == "pct":
        text = f"{float(value) * 100:.1f}%"
    elif kind == "num":
        text = f"{float(value):.2f}"
    elif kind == "int":
        text = str(int(value))
    elif kind == "date":
        text = str(pd.Timestamp(value).date())
    else:
        text = str(value)
    return text.replace(".", ",") if decimal == "," and kind in {"pct", "num"} else text


def build_output(section_specs: list[tuple[str, list[tuple[str, str, str]]]], *, decimal: str = ".") -> pd.DataFrame:
    rows = []
    index = []
    for section, metrics in section_specs:
        for metric, field, kind in metrics:
            rows.append([format_value(trio_reader.loc[idx, field], kind, decimal=decimal) for idx in PORTFOLIO_ORDER])
            index.append((section, metric))
    return pd.DataFrame(
        rows,
        index=pd.MultiIndex.from_tuples(index, names=["Section", "Metric"]),
        columns=[DISPLAY_NAMES[idx] for idx in PORTFOLIO_ORDER],
    )


paper = build_output([("Return vs. risk", RETURN_SECTION)])
paper_de = build_output([("Return vs. risk", RETURN_SECTION)], decimal=",")
thesis = build_output(
    [
        ("Return vs. risk", THESIS_RETURN),
        ("Raw market model", RAW_MODEL_SECTION),
        ("Window and provenance", WINDOW_SECTION),
    ]
)
thesis_de = build_output(
    [
        ("Return vs. risk", THESIS_RETURN),
        ("Raw market model", RAW_MODEL_SECTION),
        ("Window and provenance", WINDOW_SECTION),
    ],
    decimal=",",
)


def to_markdown(df: pd.DataFrame) -> str:
    out = [
        "| Metric | " + " | ".join(df.columns) + " |",
        "|---|" + "---|" * len(df.columns),
    ]
    current = None
    for (section, metric), row in df.iterrows():
        if section != current:
            out.append(f"| **{section}** |" + " |" * len(df.columns))
            current = section
        out.append(f"| {metric} | " + " | ".join(str(value) for value in row) + " |")
    return "\n".join(out) + "\n"


def to_latex(df: pd.DataFrame, caption: str, label: str) -> str:
    head = " & ".join(tex_escape(value) for value in ("Metric", *df.columns)) + f" {LATEX_NL}"
    lines = [
        "\\begin{table}[htbp]",
        "  \\centering",
        f"  \\caption{{{tex_escape(caption)}}}",
        f"  \\label{{{tex_escape(label)}}}",
        "  \\begin{tabular}{l" + "r" * len(df.columns) + "}",
        "    \\toprule",
        "    " + head,
        "    \\midrule",
    ]
    current = None
    for (section, metric), row in df.iterrows():
        if section != current:
            if current is not None:
                lines.append("    \\addlinespace")
            lines.append(
                f"    \\multicolumn{{{len(df.columns) + 1}}}{{l}}{{\\textit{{{tex_escape(section)}}}}} {LATEX_NL}"
            )
            current = section
        cells = " & ".join(tex_escape(value) for value in row)
        lines.append(f"    {tex_escape(metric)} & {cells} {LATEX_NL}")
    lines += ["    \\bottomrule", "  \\end{tabular}", "\\end{table}"]
    return "\n".join(lines) + "\n"


csv_provenance = f"# source_table={TRIO_SOURCE_PATH} sha256={TRIO_SOURCE_SHA256}"
md_provenance = f"<!-- source_table={TRIO_SOURCE_PATH} sha256={TRIO_SOURCE_SHA256} -->"
tex_provenance = f"% source_table={TRIO_SOURCE_PATH} sha256={TRIO_SOURCE_SHA256}"


def require_fresh_output(path: Path, provenance_line: str) -> None:
    if not path.exists():
        return
    text = path.read_text(encoding="utf-8")
    first_line = text.splitlines()[0] if text else ""
    assert first_line == provenance_line, (
        f"stale source hashes must block generation: {path.name} carries {first_line!r}, expected {provenance_line!r}"
    )


def write_csv_with_provenance(path: Path, frame: pd.DataFrame, *, sep: str = ",", decimal: str = ".") -> None:
    require_fresh_output(path, csv_provenance)
    with path.open("w", encoding="utf-8", newline="") as handle:
        handle.write(csv_provenance + "\n")
        frame.to_csv(handle, index=False, sep=sep, decimal=decimal, float_format="%.8f")


def write_text_with_provenance(path: Path, body: str, provenance_line: str) -> None:
    require_fresh_output(path, provenance_line)
    path.write_text(provenance_line + "\n" + body, encoding="utf-8")


def validate_tex_derivative(path: Path, frame: pd.DataFrame, caption: str, label: str) -> None:
    """Assert the generated text-mode table preserves provenance and TeX-safe dynamic fields."""
    lines = path.read_text(encoding="utf-8").splitlines()
    assert lines and lines[0] == tex_provenance, f"{path.name}: missing source-table provenance"
    body = lines[1:]
    assert body[0] == "\\begin{table}[htbp]" and body[-1] == "\\end{table}", f"{path.name}: malformed table environment"
    assert body.count("  \\begin{tabular}{l" + "r" * len(frame.columns) + "}") == 1, f"{path.name}: malformed tabular declaration"
    assert body.count("  \\end{tabular}") == 1, f"{path.name}: malformed tabular terminator"
    assert f"  \\caption{{{tex_escape(caption)}}}" in body, f"{path.name}: caption was not TeX-escaped"
    assert f"  \\label{{{tex_escape(label)}}}" in body, f"{path.name}: label was not TeX-escaped"

    dynamic_text = [*frame.columns, caption, label]
    for section, metric in frame.index:
        dynamic_text.extend((section, metric))
    dynamic_text.extend(value for row in frame.itertuples(index=False, name=None) for value in row)
    text = "\n".join(body)
    for value in dynamic_text:
        escaped = tex_escape(value)
        assert escaped in text, f"{path.name}: dynamic TeX text was not escaped: {value!r}"

    row_lines = [line for line in body if " & " in line or "\\multicolumn" in line]
    expected_row_lines = 1 + len(frame.index) + frame.index.get_level_values("Section").nunique()
    assert len(row_lines) == expected_row_lines, f"{path.name}: unexpected number of table rows"
    for line in row_lines:
        assert line.endswith(LATEX_NL) and not line.endswith(LATEX_NL + "\\"), (
            f"{path.name}: table row does not end in exactly two backslashes: {line!r}"
        )


window_text = f"{window_start} to {window_end}, {window_days} trading days"
outputs = {
    "paper": (paper, paper_de),
    "thesis": (thesis, thesis_de),
}
for name, (df_us, df_de) in outputs.items():
    flat_us = df_us.reset_index()
    flat_de = df_de.reset_index()
    write_csv_with_provenance(OUT / f"tear_sheet_{name}.csv", flat_us)
    write_csv_with_provenance(OUT / f"tear_sheet_{name}_de.csv", flat_de, sep=";", decimal=",")
    write_text_with_provenance(
        OUT / f"tear_sheet_{name}.md",
        f"**Trio tear sheet** — {window_text}\n\n" + to_markdown(df_us),
        md_provenance,
    )
    write_text_with_provenance(
        OUT / f"tear_sheet_{name}_de.md",
        f"**Trio tear sheet** — {window_text}\n\n" + to_markdown(df_de),
        md_provenance,
    )
    caption = f"Trio tear sheet ({window_text})."
    label = f"tab:tearsheet-{name}"
    caption_de = f"Trio tear sheet ({window_text})."
    label_de = f"tab:tearsheet-{name}-de"
    paper_tex = OUT / f"tear_sheet_{name}.tex"
    paper_de_tex = OUT / f"tear_sheet_{name}_de.tex"
    write_text_with_provenance(paper_tex, to_latex(df_us, caption, label), tex_provenance)
    write_text_with_provenance(paper_de_tex, to_latex(df_de, caption_de, label_de), tex_provenance)
    validate_tex_derivative(paper_tex, df_us, caption, label)
    validate_tex_derivative(paper_de_tex, df_de, caption_de, label_de)
    print(f"wrote {OUT / f'tear_sheet_{name}'}.{{csv,md,tex}} and *_de counterparts")

display(paper)
display(thesis)
print("source table:", TRIO_SOURCE_PATH)
print("source table sha256:", TRIO_SOURCE_SHA256)
print("stale source hashes block generation via the provenance first-line guard")
print("all TeX derivatives passed dynamic-text escaping and exact two-backslash row validation")

wrote /home/mc/projects/Global_Macro_AI_Factors/reports/validated_notebook_15_4/tear_sheet_paper.{csv,md,tex} and *_de counterparts
wrote /home/mc/projects/Global_Macro_AI_Factors/reports/validated_notebook_15_4/tear_sheet_thesis.{csv,md,tex} and *_de counterparts


Buy-and-hold AI macro-factor SJM de-risk
Section         Metric                                                  
Return vs. risk CAGR                   17.8%           12.8%       11.2%
                Ann. volatility        13.6%            9.3%        7.9%
                Sharpe                  1.10            1.09        1.08
                Max drawdown          -17.8%          -11.4%       -8.8%
                Calmar                  0.99            1.12        1.27
                SSR                     0.14            0.13        0.13

Buy-and-hold  \
Section               Metric                                                                     
Return vs. risk       Total return                                                      240.2%   
                      CAGR                                                               17.8%   
                      Ann. volatility                                                    13.6%   
                      Sharpe                                                              1.10   
                      Sortino                                                             1.57   
                      Max drawdown                                                      -17.8%   
                      Calmar                                                              0.99   
                      SSR                                                                 0.14   
Raw market model      Beta                                                                0.57   
                      R² (raw market model)                                               0.67   
                      Intercept (ann.)                                                    7.1%   
                      HAC t(intercept)                                                    2.92   
Window and provenance Start                                                         2019-01-03   
                      End                                                           2026-06-30   
                      Number of days                                                      1845   
                      Cash benchmark         BIL@provisional_market_total_return_fx_2026-06...   
                      Currency basis                                 legacy_mixed_local_quotes   

                                                                               AI macro-factor  \
Section               Metric                                                                     
Return vs. risk       Total return                                                      146.8%   
                      CAGR                                                               12.8%   
                      Ann. volatility                                                     9.3%   
                      Sharpe                                                              1.09   
                      Sortino                                                             1.57   
                      Max drawdown                                                      -11.4%   
                      Calmar                                                              1.12   
                      SSR                                                                 0.13   
Raw market model      Beta                                                                0.24   
                      R² (raw market model)                                               0.25   
                      Intercept (ann.)                                                    8.5%   
                      HAC t(intercept)                                                    3.14   
Window and provenance Start                                                         2019-01-03   
                      End                                                           2026-06-30   
                      Number of days                                                      1845   
                      Cash benchmark         BIL@provisional_market_total_return_fx_2026-06...   
                      Currency basis                                 legacy_mixed_local_quotes   

                                                                                   SJM de-risk  
Section               Metric                                                                    
Return vs. risk       Total return                                                      121.4%  
                      CAGR                                       

source table: /home/mc/projects/Global_Macro_AI_Factors/data/provisional_remediation/canonical_reports_devstartfix_full_20260730T154855Z/tables/tear_sheet_trio_ext2026.parquet
source table sha256: 1380f9aa2ed77e7b86154d95407e0b9fbd32d92a52f6b02af450310e780497cf
stale source hashes block generation via the provenance first-line guard
all TeX derivatives passed dynamic-text escaping and exact two-backslash row validation
